<a href="https://colab.research.google.com/github/ArunSekhar733/PDL_Car-or-Not/blob/main/Lesson_01_Car_vs_Plane_Full_Rewrite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 1 — Binary Image Classification with fastai

## Project

This notebook documents my implementation of the first lesson from the **Practical Deep Learning for Coders** course.

Instead of reproducing the original example, I trained a classifier to distinguish **cars** from **airplanes** using transfer learning.

---

## Objectives

- Build a small image dataset automatically.
- Clean the downloaded data.
- Create DataLoaders with the DataBlock API.
- Fine-tune a pretrained convolutional neural network.
- Test the trained model on previously unseen images.

Throughout the notebook I've included my own explanations of *why* each step exists rather than simply describing the code.


In [ ]:
import os
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')


!pip install -Uqq fastai 'duckduckgo_search>=6.2'

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac

## 1. Collecting the Dataset

A model can only learn patterns that exist in its training data.

For this experiment I automatically download images for two classes:

- Cars
- Airplanes

Using internet search is a quick way to prototype an image classifier, although the downloaded images usually contain duplicates, irrelevant results and low-quality samples.

In [ ]:


from duckduckgo_search import DDGS #DuckDuckGo has changed the api so we need to update
from fastcore.all import *

def search_images(keywords, max_images=200): return L(DDGS().images(keywords, max_results=max_images)).itemgot('image')
import time, json

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## 1. Collecting the Dataset

A model can only learn patterns that exist in its training data.

For this experiment I automatically download images for two classes:

- Cars
- Airplanes

Using internet search is a quick way to prototype an image classifier, although the downloaded images usually contain duplicates, irrelevant results and low-quality samples.

In [ ]:
#NB: `search_images` depends on duckduckgo.com, which doesn't always return correct responses.
#    If you get a JSON error, just try running it again (it may take a couple of tries).
urls = search_images('car photos', max_images=1)
urls[0]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

RatelimitException: https://duckduckgo.com/i.js?o=json&q=car+photos&l=us-en&vqd=4-126894393827980441845041819673321596300&p=1&f=%2C%2C%2C%2C%2C 403 Ratelimit

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
from fastdownload import download_url
dest = 'car.jpg'
download_url(urls[0], dest, show_progress=False)

from fastai.vision.all import *
im = Image.open(dest)
im.to_thumb(256,256)

## 1. Collecting the Dataset

A model can only learn patterns that exist in its training data.

For this experiment I automatically download images for two classes:

- Cars
- Airplanes

Using internet search is a quick way to prototype an image classifier, although the downloaded images usually contain duplicates, irrelevant results and low-quality samples.

In [ ]:
download_url(search_images('airplane photos', max_images=1)[0], 'plane.jpg', show_progress=False)
Image.open('plane.jpg').to_thumb(256,256)

## 1. Collecting the Dataset

A model can only learn patterns that exist in its training data.

For this experiment I automatically download images for two classes:

- Cars
- Airplanes

Using internet search is a quick way to prototype an image classifier, although the downloaded images usually contain duplicates, irrelevant results and low-quality samples.

In [ ]:
searches = 'plane','car'
path = Path('car_or_not')

for o in searches:
    dest = (path/o)
    dest.mkdir(exist_ok=True, parents=True)
    download_images(dest, urls=search_images(f'{o} photo'))
    time.sleep(5)
    resize_images(path/o, max_size=400, dest=path/o)

## 2. Cleaning the Dataset

Downloaded datasets are rarely perfect.

Before training I verify every image and remove files that are corrupted or unreadable. Spending a few minutes improving data quality usually has a much bigger impact than tweaking model parameters.

In [ ]:
failed = verify_images(get_image_files(path))
failed.map(Path.unlink)
len(failed)

## 3. Building the Data Pipeline

The DataBlock API defines the entire machine learning pipeline:

- where images come from
- how labels are assigned
- how the validation split is created
- how images are resized before training

Once configured, the DataBlock creates DataLoaders that feed batches of images into the neural network.

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path, bs=32)

dls.show_batch(max_n=6)

## 4. Creating the Model

Instead of starting with randomly initialized weights, I use **ResNet18**, a model that has already learned useful visual features from millions of images.

This approach is called **transfer learning** and is one of the reasons deep learning can work well even with relatively small datasets.

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

## 6. Inference

After training, the final step is to evaluate the model on an image it has never seen before.

The prediction returns:

- predicted class
- confidence scores
- probabilities for each category

This is the same workflow used when deploying image classifiers into real applications.

In [ ]:
is_car,_,probs = learn.predict(PILImage.create('car.jpg'))
print(f"This is a: {is_car}.")
print(f"Probability it's a car: {probs[0]:.4f}")

# Discussion

## Observations

- Transfer learning allowed the model to converge quickly.
- Even a relatively small dataset can produce good results when the classes are visually distinct.
- Dataset quality appears to be more important than simply increasing training time.

## Limitations

This is a simple proof-of-concept rather than a production-ready classifier.

Possible weaknesses include:
- search engine bias
- limited dataset diversity
- false labels
- background correlations

## Future Experiments

- Compare ResNet18 with ResNet34.
- Increase the number of training images.
- Evaluate using a confusion matrix.
- Test difficult examples such as toy cars, race cars, helicopters and vintage aircraft.

## Key Concepts Learned

- Transfer Learning
- Convolutional Neural Networks
- DataBlock API
- DataLoaders
- Fine-Tuning
- Image Inference

---

This notebook represents my own implementation and notes while learning fastai. The goal is to understand the workflow well enough to build more advanced computer vision projects in future lessons.
